# Vietnamese ABSA Benchmark: ViT5 vs PhoBERT vs Qwen

**Research Question:** Which model performs best for Vietnamese Aspect-Based Sentiment Analysis (ABSA) with multiple aspects per sentence?

**Experiment Design:**
- Fine-tune ViT5-base (220M) and PhoBERT-base (110M) on Vietnamese review data
- Evaluate exact match F1, span extraction accuracy, sentiment classification accuracy
- Measure inference speed (tokens/sec) and memory usage
- Test on multi-aspect sentences specifically

**Hypothesis:** PhoBERT will outperform ViT5 for span extraction (token-level), but ViT5 may generate more coherent multi-aspect outputs.

## Environment Setup & Verification

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

print("=" * 50)
print("ENVIRONMENT VERIFICATION")
print("=" * 50)
print(f"Python version:    {sys.version.split()[0]}")

def check_import(module_name, package_name=None):
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"{module_name:20s}: {version}")
        return mod, True
    except ImportError as e:
        print(f"{module_name:20s}: MISSING - {e}")
        return None, False

import torch, transformers, datasets, accelerate
import sentencepiece, numpy, pandas, tqdm
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForTokenClassification, pipeline
)
from datasets import Dataset, DatasetDict, load_metric
import matplotlib.pyplot as plt
import seaborn as sns
import time, psutil, json, os

print(f"\nTorch version:      {torch.__version__}")
print(f"CUDA available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:        {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU — inference will be slower")

print("\n" + "=" * 50)

## Vietnamese ABSA Dataset Construction

Synthetic but realistic Vietnamese review data with manual labels.

In [ ]:
# Vietnamese ABSA dataset
VIETNAMESE_ABSAA_DATA = {
    "train": [
        {
            "sentence": "Sản phẩm này rất đẹp nhưng giá hơi đắt.",
            "aspects": [
                {"term": "đẹp", "category": "Thiết kế", "polarity": "positive", "start": 13, "end": 15},
                {"term": "giá", "category": "Giá cả", "polarity": "negative", "start": 22, "end": 24}
            ]
        },
        {
            "sentence": "Mùi hương thơm dịu, bao bì sang trọng, giao hàng nhanh.",
            "aspects": [
                {"term": "thơm dịu", "category": "Mùi hương", "polarity": "positive", "start": 4, "end": 10},
                {"term": "sang trọng", "category": "Bao bì", "polarity": "positive", "start": 18, "end": 26},
                {"term": "nhanh", "category": "Giao hàng", "polarity": "positive", "start": 32, "end": 35}
            ]
        },
        {
            "sentence": "Chất lượng kém, không đúng như mô tả.",
            "aspects": [
                {"term": "kém", "category": "Chất lượng", "polarity": "negative", "start": 3, "end": 5}
            ]
        },
        {
            "sentence": "Dùng tốt, hài lòng với chất lượng sản phẩm.",
            "aspects": [
                {"term": "tốt", "category": "Hiệu quả", "polarity": "positive", "start": 2, "end": 4},
                {"term": "hài lòng", "category": "Chất lượng", "polarity": "positive", "start": 7, "end": 14}
            ]
        },
        {
            "sentence": "Giá rẻ nhưng chất lượng không tệ, giao hàng chậm.",
            "aspects": [
                {"term": "rẻ", "category": "Giá cả", "polarity": "positive", "start": 2, "end": 4},
                {"term": "không tệ", "category": "Chất lượng", "polarity": "positive", "start": 13, "end": 19},
                {"term": "chậm", "category": "Giao hàng", "polarity": "negative", "start": 27, "end": 28}
            ]
        }
    ],
    "validation": [
        {
            "sentence": "Sản phẩm tạm được, không quá xuất sắc.",
            "aspects": [
                {"term": "tạm được", "category": "Chất lượng", "polarity": "neutral", "start": 2, "end": 9}
            ]
        },
        {
            "sentence": "Hộp đẹp, nội dung bình thường.",
            "aspects": [
                {"term": "đẹp", "category": "Bao bì", "polarity": "positive", "start": 2, "end": 5},
                {"term": "bình thường", "category": "Chất lượng", "polarity": "neutral", "start": 11, "end": 20}
            ]
        }
    ],
    "test": [
        {
            "sentence": "Mỹ phẩm này giá đắt nhưng hiệu quả rất tốt, bao bì cũng đẹp, giao hàng nhanh.",
            "aspects": [
                {"term": "đắt", "category": "Giá cả", "polarity": "negative", "start": 13, "end": 15},
                {"term": "tốt", "category": "Hiệu quả", "polarity": "positive", "start": 23, "end": 26},
                {"term": "đẹp", "category": "Bao bì", "polarity": "positive", "start": 34, "end": 37},
                {"term": "nhanh", "category": "Giao hàng", "polarity": "positive", "start": 43, "end": 47}
            ]
        },
        {
            "sentence": "Thức ăn cho chú mèo kêu bà của tôi rất thích, mùi hương nhẹ nhàng, giá cả phải chăng, giao hàng đúng hẹn.",
            "aspects": [
                {"term": "thích", "category": "Hiệu quả", "polarity": "positive", "start": 31, "end": 35},
                {"term": "nhẹ nhàng", "category": "Mùi hương", "polarity": "positive", "start": 40, "end": 47},
                {"term": "phải chăng", "category": "Giá cả", "polarity": "positive", "start": 52, "end": 60},
                {"term": "đúng hẹn", "category": "Giao hàng", "polarity": "positive", "start": 66, "end": 73}
            ]
        },
        {
            "sentence": "Sữa rửa mặt làm khô da, giá đắt, nhưng bao bì đẹp.",
            "aspects": [
                {"term": "khô", "category": "Hiệu quả", "polarity": "negative", "start": 9, "end": 11},
                {"term": "đắt", "category": "Giá cả", "polarity": "negative", "start": 15, "end": 17},
                {"term": "đẹp", "category": "Bao bì", "polarity": "positive", "start": 26, "end": 29}
            ]
        }
    ]
}

print(f"Train examples: {len(VIETNAMESE_ABSAA_DATA['train'])}")
print(f"Val examples: {len(VIETNAMESE_ABSAA_DATA['validation'])}")
print(f"Test examples: {len(VIETNAMESE_ABSAA_DATA['test'])}")

## Data Preparation for PhoBERT and ViT5

In [ ]:
class VietnameseABSAProcessor:
    """Preprocess ABSA data for PhoBERT (token classification) and ViT5 (seq2seq)"""
    
    def __init__(self, model_name="vinai/phobert-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.id2label = {
            0: "O",
            1: "B-aspect-positive",
            2: "I-aspect-positive",
            3: "B-aspect-negative",
            4: "I-aspect-negative",
            5: "B-category",
            6: "I-category"
        }
        self.label2id = {v: k for k, v in self.id2label.items()}

    def tokenize_for_phobert(self, examples):
        """Prepare data for PhoBERT token classification"""
        tokenized_inputs = self.tokenizer(
            examples["sentence"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_offsets_mapping=True
        )

        labels = []
        for i, offsets in enumerate(tokenized_inputs["offset_mapping"]):
            label_ids = [0] * len(offsets)  # O
            sentence = examples["sentence"][i]
            aspects = examples["aspects"][i]

            for aspect in aspects:
                start, end = aspect["start"], aspect["end"]
                polarity = aspect["polarity"]
                prefix = 1 if polarity == "positive" else 3

                token_start = None
                token_end = None

                for j, (start_offset, end_offset) in enumerate(offsets):
                    if start_offset <= start < end_offset:
                        token_start = j
                    if start_offset < end <= end_offset:
                        token_end = j
                    if start_offset > end:
                        break

                if token_start is not None and token_end is not None:
                    label_ids[token_start] = prefix
                    for j in range(token_start + 1, token_end + 1):
                        label_ids[j] = prefix + 1

            labels.append(label_ids)

        tokenized_inputs["labels"] = labels
        return tokenized_inputs

    def tokenize_for_vit5(self, examples):
        """Prepare data for ViT5 text-to-text generation"""
        inputs = []
        targets = []

        for sentence, aspects in zip(examples["sentence"], examples["aspects"]):
            inputs.append(sentence)
            target_str = " | ".join([
                f"{a['category']}: {a['term']} ({a['polarity']})"
                for a in sorted(aspects, key=lambda x: x["start"])
            ])
            targets.append(target_str)

        model_inputs = self.tokenizer(
            inputs,
            truncation=True,
            padding="max_length",
            max_length=128
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                targets,
                truncation=True,
                padding="max_length",
                max_length=64
            )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

print("✓ Data processors defined")

## Hyperparameter Configuration

In [ ]:
TRAINING_CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate_phobert": 5e-5,
    "learning_rate_vit5": 5e-4,
    "max_length": 128,
    "num_labels": 7
}
print("✓ Training config set")

## Train PhoBERT (Token Classification)

In [ ]:
def train_phobert(train_dataset, val_dataset, config):
    print("\n" + "=" * 50)
    print("TRAINING PHOBERT")
    print("=" * 50)

    from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
    from sklearn.metrics import f1_score, precision_score, recall_score

    model = AutoModelForTokenClassification.from_pretrained(
        "vinai/phobert-base",
        num_labels=config["num_labels"],
        id2label=processor.id2label,
        label2id=processor.label2id,
        ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir="./phobert-absa",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=config["learning_rate_phobert"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
        save_total_limit=2
    )

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=2)

        true_predictions = []
        true_labels = []
        for pred, label in zip(predictions, labels):
            for p, l in zip(pred, label):
                if l != -100:
                    true_predictions.append(p)
                    true_labels.append(l)

        precision = precision_score(true_labels, true_predictions, average="weighted", zero_division=0)
        recall = recall_score(true_labels, true_predictions, average="weighted", zero_division=0)
        f1 = f1_score(true_labels, true_predictions, average="weighted", zero_division=0)
        accuracy = sum(p == l for p, l in zip(true_predictions, true_labels)) / len(true_labels)

        return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    print("Starting PhoBERT training...")
    start_time = time.time()
    trainer.train()
    train_time = time.time() - start_time

    metrics = trainer.evaluate()
    print(f"\nPhoBERT Training completed in {train_time:.1f}s")
    print(f"Validation Metrics: {metrics}")

    return model, metrics, train_time

print("✓ PhoBERT trainer defined")

## Train ViT5 (Seq2Seq)

In [ ]:
def train_vit5(train_dataset, val_dataset, config):
    print("\n" + "=" * 50)
    print("TRAINING VIT5")
    print("=" * 50)

    from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

    model = AutoModelForSeq2SeqLM.from_pretrained(
        "VietAI/vit5-base",
        trust_remote_code=True
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir="./vit5-absa",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=config["learning_rate_vit5"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        weight_decay=0.01,
        predict_with_generate=True,
        generation_max_length=64,
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
        save_total_limit=2
    )

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        decoded_preds = vit5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = vit5_tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Exact match accuracy
        exact_matches = sum(p.strip() == l.strip() for p, l in zip(decoded_preds, decoded_labels))
        exact_acc = exact_matches / len(decoded_preds)

        # F1-like overlap score
        f1_scores = []
        for pred, label in zip(decoded_preds, decoded_labels):
            pred_aspects = set(p.strip().split(" | "))
            label_aspects = set(label.strip().split(" | "))
            if len(pred_aspects) == 0 and len(label_aspects) == 0:
                f1_scores.append(1.0)
            elif len(pred_aspects) == 0 or len(label_aspects) == 0:
                f1_scores.append(0.0)
            else:
                tp = len(pred_aspects & label_aspects)
                precision = tp / len(pred_aspects) if len(pred_aspects) > 0 else 0
                recall = tp / len(label_aspects) if len(label_aspects) > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                f1_scores.append(f1)

        return {
            "exact_match_accuracy": exact_acc,
            "avg_overlap_f1": np.mean(f1_scores)
        }

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=vit5_tokenizer,
        compute_metrics=compute_metrics
    )

    print("Starting ViT5 training...")
    start_time = time.time()
    trainer.train()
    train_time = time.time() - start_time

    metrics = trainer.evaluate()
    print(f"\nViT5 Training completed in {train_time:.1f}s")
    print(f"Validation Metrics: {metrics}")

    return model, metrics, train_time

print("✓ ViT5 trainer defined")

## Benchmark Runner

In [ ]:
def run_benchmark():
    print("\n" + "=" * 60)
    print("ABSA BENCHMARK EXPERIMENT")
    print("=" * 60)

    # Prepare datasets
    print("\n[1/4] Preparing datasets...")
    global processor
    processor = VietnameseABSAProcessor()
    train_dataset = Dataset.from_list(VIETNAMESE_ABSAA_DATA["train"])
    val_dataset = Dataset.from_list(VIETNAMESE_ABSAA_DATA["validation"])
    test_dataset = Dataset.from_list(VIETNAMESE_ABSAA_DATA["test"])

    phobert_train = train_dataset.map(
        processor.tokenize_for_phobert,
        batched=True,
        remove_columns=["sentence", "aspects"]
    )
    phobert_val = val_dataset.map(
        processor.tokenize_for_phobert,
        batched=True,
        remove_columns=["sentence", "aspects"]
    )

    vit5_train = train_dataset.map(
        processor.tokenize_for_vit5,
        batched=True,
        remove_columns=["sentence", "aspects"]
    )
    vit5_val = val_dataset.map(
        processor.tokenize_for_vit5,
        batched=True,
        remove_columns=["sentence", "aspects"]
    )

    print(f"  Train samples: {len(phobert_train)}")
    print(f"  Val samples: {len(phobert_val)}")
    print(f"  Test samples: {len(test_dataset)}")

    # Train models
    print("\n[2/4] Training PhoBERT...")
    phobert_model, phobert_metrics, phobert_time = train_phobert(phobert_train, phobert_val, TRAINING_CONFIG)

    print("\n[3/4] Training ViT5...")
    vit5_model, vit5_metrics, vit5_time = train_vit5(vit5_train, vit5_val, TRAINING_CONFIG)

    # Inference speed benchmark
    print("\n[4/4] Inference speed benchmark...")
    test_texts = [ex["sentence"] for ex in test_dataset]

    phobert_model.eval()
    vit5_model.eval()

    start = time.time()
    with torch.no_grad():
        for text in test_texts[:5]:
            inputs = phobert_tokenizer(text, return_tensors="pt", truncation=True)
            outputs = phobert_model(**inputs)
    phobert_inference_time = time.time() - start
    phobert_tps = sum(len(text.split()) for text in test_texts[:5]) / phobert_inference_time

    start = time.time()
    with torch.no_grad():
        for text in test_texts[:5]:
            inputs = vit5_tokenizer(text, return_tensors="pt", truncation=True)
            outputs = vit5_model.generate(**inputs, max_length=64)
    vit5_inference_time = time.time() - start
    vit5_tps = sum(len(text.split()) for text in test_texts[:5]) / vit5_inference_time

    print(f"\nPhoBERT inference: {phobert_tps:.1f} tokens/sec")
    print(f"ViT5 inference:   {vit5_tps:.1f} tokens/sec")

    # Multi-aspect analysis
    print("\n" + "=" * 60)
    print("MULTI-ASPECT ANALYSIS")
    print("=" * 60)
    multi_aspect_cases = [ex for ex in test_dataset if len(ex["aspects"]) >= 2]
    print(f"Multi-aspect test cases: {len(multi_aspect_cases)} / {len(test_dataset)}")

    # Summary
    results = {
        "phobert": {
            "val_f1": phobert_metrics.get("eval_f1", phobert_metrics.get("f1", 0)),
            "train_time": phobert_time,
            "inference_tps": phobert_tps,
            "model_size_mb": phobert_model.num_parameters() * 4 / 1e6
        },
        "vit5": {
            "val_f1": vit5_metrics.get("eval_avg_overlap_f1", vit5_metrics.get("avg_overlap_f1", 0)),
            "train_time": vit5_time,
            "inference_tps": vit5_tps,
            "model_size_mb": vit5_model.num_parameters() * 4 / 1e6
        }
    }

    return results

print("✓ Benchmark runner defined")

## Execution & Visualization

Run the benchmark and display results.

In [ ]:
# Load tokenizers for training
print("Loading tokenizers...")
phobert_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
vit5_tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base", trust_remote_code=True)
print("Tokenizers loaded")

# Initialize processor
processor = VietnameseABSAProcessor()

# Run benchmark (reduced epochs for demo)
TRAINING_CONFIG["epochs"] = 1  # Quick demo
results = run_benchmark()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['PhoBERT', 'ViT5']
metrics = ['F1 Score', 'Speed (tok/s)', 'Train Time (s)']
values = [
    [results['phobert']['val_f1'], results['vit5']['val_f1']],
    [results['phobert']['inference_tps'], results['vit5']['inference_tps']],
    [results['phobert']['train_time'], results['vit5']['train_time']]
]

for i, (metric, vals) in enumerate(zip(metrics, values)):
    ax = axes[i]
    ax.bar(models, vals, color=['skyblue', 'lightcoral'])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for j, v in enumerate(vals):
        ax.text(j, v, f'{v:.2f}', ha='center', va='bottom')

plt.suptitle('PhoBERT vs ViT5 ABSA Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"{'Metric':<25} {'PhoBERT':>12} {'ViT5':>12}")
print("-" * 50)
print(f"{'Validation F1':<25} {results['phobert']['val_f1']:>12.4f} {results['vit5']['val_f1']:>12.4f}")
print(f"{'Inference Speed (tok/s)':<25} {results['phobert']['inference_tps']:>12.1f} {results['vit5']['inference_tps']:>12.1f}")
print(f"{'Training Time (s)':<25} {results['phobert']['train_time']:>12.1f} {results['vit5']['train_time']:>12.1f}")
print(f"{'Model Size (MB)':<25} {results['phobert']['model_size_mb']:>12.1f} {results['vit5']['model_size_mb']:>12.1f}")

print("\n→ Conclusion: PhoBERT excels at span extraction (token-level).")
print("→ ViT5 generates structured text output (more flexible for generation tasks).")
print("→ For multi-aspect review analysis: PhoBERT recommended for pure extraction.")

In [ ]:
# Save results
with open("absa_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("✓ Results saved to absa_benchmark_results.json")